In [7]:
%matplotlib qt
%reload_ext autoreload
%autoreload 2
import pl_spec_python as pl_spec
import plotter
import sgd
import filter as fil
sgd.sgd_init()

<'USBInstrument'('USB0::0xF4ED::0xEE3A::SDG10GAD1R1771::0::INSTR')>

In [3]:
import plotter



## Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `foldername` | str | Base name of the folder where data will be saved. |
| `scan_type` | str | Subfolder category. Options: `"coarse"` or `"fine"`. |
| `xdim`, `ydim` | float | Scan area width/height in um. Set to `0` for single-point. |
| `dx`, `dy` | float | Step size in um. |
| `center` | tuple | `(x, y)` centre of scan in um. |
| `grating` | int | `150` (150 g/mm) or `600` (600 g/mm). |
| `exposure_time` | float | Integration time per point in seconds. |
| `center_wavelength` | int | Spectrometer centre wavelength in nm. |
| `classification_threshold` | float | Min fraction of laser peak for emitter classification. |
| `current_user` | str | User for Telegram focus alerts. Options: `"shuhul"`, `"kristina"`, `"holland"`. |

In [13]:
foldername = '20260604-PLSPC-EL-BulkAnneal5262026-Ch3-f9-POSTPLASMA-450uW-2s-1'
scan_type  = 'coarse'

pl_spec.pl_spec_lf(
    xdim=12, ydim=12, dx=0.5, dy=0.5, center=(0, 0),
    grating=150,
    exposure_time=2,
    center_wavelength=700,
    classification_threshold=1.05,
    foldername=foldername,
    scan_type=scan_type,
    current_user='holland'
)

plotter.plot_heatmap_manual(foldername, scan_type)

Connecting to LightField...
LightField bridge already running.
Getting wavelengths and setting up...
Starting scan...
Sgd on!


Acquiring x=6.00, y=6.00: 100%|██████████| 625/625 [37:05<00:00,  3.56s/it]  


Sgd off!
Scan complete, classifying data...
Found 8 distinct emitters.
Classification complete!


In [18]:
# Goto
sgd.goto(0, 0)

Sgd on!
Moving to (0 um, 0 um)
Done moving!


In [3]:
# End goto
sgd.home()

Moving to (0 um, 0 um)
Done moving!
Sgd off!


In [16]:
foldername = '20260604-PLSPC-HT-Ch4-f1-500uW-1s-fullauto'
scan_type  = 'coarse'
plotter.plot_heatmap_manual(foldername, scan_type)

In [4]:
FOLDERNAME   = '20260608-PLSPC-HT-Ch4-f2-500uW-1s-fullauto'
DATA_FOLDER  = 'data'
plotter.plot_heatmap_manual(FOLDERNAME, 'coarse', data_folder=DATA_FOLDER)

In [11]:
import filter as fil
print(fil.flipper)

Thorlabs.MotionControl.FilterFlipperCLI.FilterFlipper


In [12]:

  fil.filter_init()
  fil.filter_on()
  fil.flip_down()
  print(fil.flipper)
  if fil.flipper is not None:
      fil.filter_off()


Building device list for filter...
Connecting to Filter Flipper (Serial: 37010764)...
Connecting to Filter Rotation Stage (Serial: 27600279)...
Initializing filter...
Done initializing filter!
Turning filter on...
Done turning on!
Flipping down...
Done Flipping!
Thorlabs.MotionControl.FilterFlipperCLI.FilterFlipper
Turning filter off...
Done turning off!


## Standalone G2

Run a G2 measurement on its own (no scans). Useful for checking the PicoHarp setup and comparing the live count rates against the PicoHarp software for the same alignment before committing to a full acquisition.

In [ ]:
import picoharp
import g2 as g2mod
import os
from datetime import datetime

DATA_FOLDER       = 'data'
G2_TARGET_RECORDS = 1_000_000
G2_TIME_NS        = 100.0
G2_TIMEBIN_NS     = 0.25

picoharp.ph_init()


In [ ]:
# Live count rates -- compare these against the PicoHarp software
# for the same alignment before acquiring.
for _ in range(4):
    r0, r1 = picoharp.get_count_rates()
    print(f'Ch0: {r0:.2e} cps   Ch1: {r1:.2e} cps')


In [ ]:
# Acquire and run the g2 analysis
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
g2_folder = os.path.join(DATA_FOLDER, f'g2_standalone_{timestamp}')

npz_path = picoharp.ph_acquire(G2_TARGET_RECORDS, out_folder=g2_folder)

if npz_path:
    g2_result = g2mod.run(npz_path, out_folder=g2_folder,
                           g2time_ns=G2_TIME_NS, timebin_ns=G2_TIMEBIN_NS)
    if g2_result['popt'] is not None:
        g2_0 = 1 - g2_result['popt'][1]
        print(f'g²(0) = {g2_0:.3f}')
    else:
        print('g² fit did not converge.')
else:
    print('G2 acquisition returned no data.')


In [ ]:
# Disconnect when done
picoharp.ph_close()
